In [1]:
# Step 1: torchaudio hata do (transformers crash se bachne ke liye)
!pip uninstall -y torchaudio -q

# Step 2: onnxruntime-gpu - CUDA 12.x wala default build (stable, well-tested)
!pip install -q onnxruntime-gpu==1.20.1

# Step 3: TensorRT - explicitly CUDA 12 variant (generic "tensorrt" naam cu13 khींch leta hai is system pe)
!pip install -q tensorrt-cu12==10.7.0 tensorrt-cu12-libs==10.7.0 tensorrt-cu12-bindings==10.7.0 --extra-index-url https://pypi.nvidia.com

# Step 4: transformers (koi version issue nahi is se)
!pip install -q transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.5/291.5 MB 6.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 GB 23.3 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 222.0 MB/s eta 0:00:00


In [2]:
!find / -name "libnvinfer.so*" 2>/dev/null
!find / -name "libcudart.so*" 2>/dev/null

/usr/local/lib/python3.12/dist-packages/tensorrt_libs/libnvinfer.so.10
/usr/local/lib/python3.12/dist-packages/nvidia/cuda_runtime/lib/libcudart.so.12
/usr/local/cuda-12.8/targets/x86_64-linux/lib/libcudart.so.12
/usr/local/cuda-12.8/targets/x86_64-linux/lib/libcudart.so.12.8.90
/usr/local/cuda-12.8/targets/x86_64-linux/lib/libcudart.so


In [3]:
import torch
import onnxruntime as ort

print("PyTorch CUDA available:", torch.cuda.is_available())
print("ONNX Runtime providers:", ort.get_available_providers())

PyTorch CUDA available: True
ONNX Runtime providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [4]:
from transformers import ViTForImageClassification
import torch

model_name = "google/vit-base-patch16-224"
model = ViTForImageClassification.from_pretrained(model_name)
model.eval()
model = model.to("cuda")

print("Model loaded. Total parameters:", sum(p.numel() for p in model.parameters()))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Model loaded. Total parameters: 86567656


In [7]:
pip install onnxscript -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 12.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 14.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [8]:
model_cpu = model.to("cpu")
model_cpu.eval()

dummy_input = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model_cpu,
    dummy_input,
    "vit_base.onnx",
    export_params=True,
    opset_version=17,
    input_names=["pixel_values"],
    output_names=["logits"],
    dynamic_axes={
        "pixel_values": {0: "batch_size"},
        "logits": {0: "batch_size"}
    }
)

model = model.to("cuda")
print("ONNX export done.")

/tmp/ipykernel_58/110138947.py:6: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0715 15:27:45.773000 58 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0715 15:27:46.837000 58 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, ali

[torch.onnx] Obtain model graph for `ViTForImageClassification([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ViTForImageClassification([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
ONNX export done.


In [10]:
import os

tensorrt_lib_path = "/usr/local/lib/python3.12/dist-packages/tensorrt_libs"

os.environ["LD_LIBRARY_PATH"] = tensorrt_lib_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")

print("Updated LD_LIBRARY_PATH:", os.environ["LD_LIBRARY_PATH"])

Updated LD_LIBRARY_PATH: /usr/local/lib/python3.12/dist-packages/tensorrt_libs:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:/usr/local/cuda/lib64


In [12]:
!ln -sf /usr/local/lib/python3.12/dist-packages/tensorrt_libs/libnvinfer.so.10 /usr/lib/libnvinfer.so.10
!ln -sf /usr/local/lib/python3.12/dist-packages/tensorrt_libs/libnvinfer_plugin.so.11 /usr/lib/libnvinfer_plugin.so.11
!ldconfig

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/li

In [14]:
!find / -name "libnvinfer_plugin*" 2>/dev/null

/usr/local/lib/python3.12/dist-packages/tensorrt_libs/libnvinfer_plugin.so.10


In [15]:
!ln -sf /usr/local/lib/python3.12/dist-packages/tensorrt_libs/libnvinfer_plugin.so.10 /usr/lib/libnvinfer_plugin.so.10
!ldconfig 2>/dev/null

In [18]:
import os
import glob

tensorrt_libs_dir = "/usr/local/lib/python3.12/dist-packages/tensorrt_libs"
target_dir = "/usr/lib"

# Sab .so files dhoondo tensorrt_libs folder mein
so_files = glob.glob(os.path.join(tensorrt_libs_dir, "*.so*"))

print(f"Found {len(so_files)} library files")

for so_file in so_files:
    filename = os.path.basename(so_file)
    target_path = os.path.join(target_dir, filename)
    if not os.path.exists(target_path):
        os.symlink(so_file, target_path)
        print(f"Linked: {filename}")

!ldconfig 2>/dev/null
print("Done.")

Found 5 library files
Linked: libnvinfer_builder_resource.so.10.7.0
Linked: libnvonnxparser.so.10
Linked: libnvinfer_builder_resource_win.so.10.7.0
Done.


In [19]:
session_trt = ort.InferenceSession("vit_base.onnx", providers=providers)
print("Active provider:", session_trt.get_providers())

Active provider: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [20]:
import numpy as np
import time

warmup_input = np.random.randn(1, 3, 224, 224).astype(np.float32)

print("Building TensorRT engine (first run, may take a few minutes)...")
start = time.perf_counter()
_ = session_trt.run(None, {"pixel_values": warmup_input})
print(f"First run (includes engine build): {(time.perf_counter() - start)*1000:.2f}ms")

for _ in range(10):
    _ = session_trt.run(None, {"pixel_values": warmup_input})

print("Warmup done.")

Building TensorRT engine (first run, may take a few minutes)...
First run (includes engine build): 37235.62ms
Warmup done.


In [21]:
def benchmark_tensorrt(batch_size, num_runs=50):
    input_data = np.random.randn(batch_size, 3, 224, 224).astype(np.float32)
    
    # Naya batch size = naya engine shape, pehla call compile karega
    _ = session_trt.run(None, {"pixel_values": input_data})
    
    latencies = []
    for _ in range(num_runs):
        start = time.perf_counter()
        _ = session_trt.run(None, {"pixel_values": input_data})
        end = time.perf_counter()
        latencies.append((end - start) * 1000)
    
    return {
        "batch_size": batch_size,
        "mean_ms": np.mean(latencies),
        "p50_ms": np.percentile(latencies, 50),
        "p95_ms": np.percentile(latencies, 95),
        "p99_ms": np.percentile(latencies, 99),
    }

batch_sizes = [1, 4, 8, 16]
trt_results = []

for bs in batch_sizes:
    result = benchmark_tensorrt(bs)
    trt_results.append(result)
    print(f"Batch {bs:2d} | Mean: {result['mean_ms']:.2f}ms | P50: {result['p50_ms']:.2f}ms | P95: {result['p95_ms']:.2f}ms | P99: {result['p99_ms']:.2f}ms")

Batch  1 | Mean: 5.14ms | P50: 5.62ms | P95: 5.67ms | P99: 5.74ms
Batch  4 | Mean: 8.46ms | P50: 8.38ms | P95: 8.92ms | P99: 9.15ms
Batch  8 | Mean: 15.18ms | P50: 15.30ms | P95: 15.67ms | P99: 15.87ms
Batch 16 | Mean: 32.54ms | P50: 32.59ms | P95: 34.06ms | P99: 34.50ms


In [22]:
import torch
import time
import numpy as np

model.eval()

# Warmup
warmup_input = torch.randn(1, 3, 224, 224).to("cuda")
with torch.no_grad():
    for _ in range(10):
        _ = model(warmup_input)
torch.cuda.synchronize()

def benchmark_pytorch(batch_size, num_runs=50):
    input_tensor = torch.randn(batch_size, 3, 224, 224).to("cuda")
    latencies = []
    with torch.no_grad():
        for _ in range(num_runs):
            torch.cuda.synchronize()
            start = time.perf_counter()
            _ = model(input_tensor)
            torch.cuda.synchronize()
            end = time.perf_counter()
            latencies.append((end - start) * 1000)
    return {
        "batch_size": batch_size,
        "mean_ms": np.mean(latencies),
        "p50_ms": np.percentile(latencies, 50),
        "p95_ms": np.percentile(latencies, 95),
        "p99_ms": np.percentile(latencies, 99),
    }

batch_sizes = [1, 4, 8, 16]
pytorch_results = []
for bs in batch_sizes:
    result = benchmark_pytorch(bs)
    pytorch_results.append(result)
    print(f"Batch {bs:2d} | Mean: {result['mean_ms']:.2f}ms | P50: {result['p50_ms']:.2f}ms | P95: {result['p95_ms']:.2f}ms | P99: {result['p99_ms']:.2f}ms")

Batch  1 | Mean: 14.88ms | P50: 14.52ms | P95: 18.70ms | P99: 22.05ms
Batch  4 | Mean: 47.07ms | P50: 46.90ms | P95: 47.74ms | P99: 52.82ms
Batch  8 | Mean: 80.73ms | P50: 80.45ms | P95: 80.81ms | P99: 96.25ms
Batch 16 | Mean: 166.85ms | P50: 166.88ms | P95: 169.63ms | P99: 170.04ms


In [23]:
providers_cuda_only = ["CUDAExecutionProvider", "CPUExecutionProvider"]
session_cuda = ort.InferenceSession("vit_base.onnx", providers=providers_cuda_only)
print("Active provider:", session_cuda.get_providers())

warmup_input_np = np.random.randn(1, 3, 224, 224).astype(np.float32)
for _ in range(10):
    _ = session_cuda.run(None, {"pixel_values": warmup_input_np})

def benchmark_onnx_cuda(batch_size, num_runs=50):
    input_data = np.random.randn(batch_size, 3, 224, 224).astype(np.float32)
    latencies = []
    for _ in range(num_runs):
        start = time.perf_counter()
        _ = session_cuda.run(None, {"pixel_values": input_data})
        end = time.perf_counter()
        latencies.append((end - start) * 1000)
    return {
        "batch_size": batch_size,
        "mean_ms": np.mean(latencies),
        "p50_ms": np.percentile(latencies, 50),
        "p95_ms": np.percentile(latencies, 95),
        "p99_ms": np.percentile(latencies, 99),
    }

onnx_cuda_results = []
for bs in batch_sizes:
    result = benchmark_onnx_cuda(bs)
    onnx_cuda_results.append(result)
    print(f"Batch {bs:2d} | Mean: {result['mean_ms']:.2f}ms | P50: {result['p50_ms']:.2f}ms | P95: {result['p95_ms']:.2f}ms | P99: {result['p99_ms']:.2f}ms")

Active provider: ['CUDAExecutionProvider', 'CPUExecutionProvider']
Batch  1 | Mean: 15.43ms | P50: 15.51ms | P95: 15.74ms | P99: 15.93ms
Batch  4 | Mean: 46.89ms | P50: 47.00ms | P95: 47.12ms | P99: 47.35ms
Batch  8 | Mean: 94.77ms | P50: 94.81ms | P95: 95.96ms | P99: 98.09ms
Batch 16 | Mean: 189.68ms | P50: 189.36ms | P95: 191.37ms | P99: 199.73ms


In [24]:
# Same input, teeno models se output nikalo, compare karo

test_input_torch = torch.randn(1, 3, 224, 224).to("cuda")
test_input_np = test_input_torch.cpu().numpy()

with torch.no_grad():
    pytorch_output = model(test_input_torch).logits.cpu().numpy()

onnx_output = session_cuda.run(None, {"pixel_values": test_input_np})[0]
trt_output = session_trt.run(None, {"pixel_values": test_input_np})[0]

# Predicted class (argmax) compare karo - sabse important cheez
print("PyTorch predicted class:", np.argmax(pytorch_output))
print("ONNX predicted class:", np.argmax(onnx_output))
print("TensorRT predicted class:", np.argmax(trt_output))

# Numerical difference bhi dekho
print("\nMean abs difference (PyTorch vs ONNX):", np.mean(np.abs(pytorch_output - onnx_output)))
print("Mean abs difference (PyTorch vs TensorRT):", np.mean(np.abs(pytorch_output - trt_output)))

PyTorch predicted class: 868
ONNX predicted class: 868
TensorRT predicted class: 868

Mean abs difference (PyTorch vs ONNX): 4.835837e-07
Mean abs difference (PyTorch vs TensorRT): 0.002214298


In [25]:
from datasets import load_dataset

# ImageNet-1k ka ek chhota validation sample - "tiny-imagenet" ya validation split ka subset
# Ye dataset already labeled hai, ImageNet class IDs ke saath
dataset = load_dataset("zh-plus/tiny-imagenet", split="valid[:100]")

print("Total samples loaded:", len(dataset))
print("Sample entry:", dataset[0].keys())

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-1359597a978bc4(…):   0%|          | 0.00/146M [00:00<?, ?B/s]

data/valid-00000-of-00001-70d52db3c749a9(…):   0%|          | 0.00/14.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Total samples loaded: 100
Sample entry: dict_keys(['image', 'label'])


In [26]:
from transformers import ViTImageProcessor
import numpy as np
import torch

processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

def preprocess_image(pil_image):
    # Processor image ko resize, normalize, aur tensor mein convert karta hai
    inputs = processor(images=pil_image, return_tensors="pt")
    return inputs["pixel_values"]  # shape: [1, 3, 224, 224]

# Test - ek sample process karke dekho
sample_image = dataset[0]["image"].convert("RGB")  # RGB force karo (kuch images grayscale ho sakti hain)
processed = preprocess_image(sample_image)
print("Processed shape:", processed.shape)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Processed shape: torch.Size([1, 3, 224, 224])


In [27]:
processed_images = []

for i in range(len(dataset)):
    img = dataset[i]["image"].convert("RGB")
    processed = preprocess_image(img)
    processed_images.append(processed)

# Sab ko stack karo ek single tensor mein
all_images_tensor = torch.cat(processed_images, dim=0)
print("Final batch shape:", all_images_tensor.shape)  # expect: [100, 3, 224, 224]

Final batch shape: torch.Size([100, 3, 224, 224])


In [28]:
import numpy as np
import torch

all_images_np = all_images_tensor.numpy()

# PyTorch predictions
pytorch_preds = []
model.eval()
with torch.no_grad():
    for i in range(len(all_images_tensor)):
        single_input = all_images_tensor[i:i+1].to("cuda")
        output = model(single_input).logits.cpu().numpy()
        pytorch_preds.append(np.argmax(output))

# ONNX Runtime (CUDA) predictions
onnx_preds = []
for i in range(len(all_images_np)):
    single_input = all_images_np[i:i+1]
    output = session_cuda.run(None, {"pixel_values": single_input})[0]
    onnx_preds.append(np.argmax(output))

# TensorRT predictions
trt_preds = []
for i in range(len(all_images_np)):
    single_input = all_images_np[i:i+1]
    output = session_trt.run(None, {"pixel_values": single_input})[0]
    trt_preds.append(np.argmax(output))

pytorch_preds = np.array(pytorch_preds)
onnx_preds = np.array(onnx_preds)
trt_preds = np.array(trt_preds)

# Match rates nikalo
onnx_match = np.mean(pytorch_preds == onnx_preds) * 100
trt_match = np.mean(pytorch_preds == trt_preds) * 100

print(f"PyTorch vs ONNX Runtime - Prediction match: {onnx_match:.1f}% ({np.sum(pytorch_preds == onnx_preds)}/100)")
print(f"PyTorch vs TensorRT     - Prediction match: {trt_match:.1f}% ({np.sum(pytorch_preds == trt_preds)}/100)")

# Kaunse samples mismatch hue (agar hon), dekho
mismatches = np.where(pytorch_preds != trt_preds)[0]
print(f"\nMismatched sample indices (PyTorch vs TensorRT): {mismatches.tolist()}")

PyTorch vs ONNX Runtime - Prediction match: 100.0% (100/100)
PyTorch vs TensorRT     - Prediction match: 100.0% (100/100)

Mismatched sample indices (PyTorch vs TensorRT): []


In [29]:
import json
import shutil
import os
from datetime import datetime

# Step 1: Ek results folder banao jahan sab kuch collect hoga
output_dir = "inferbench_kaggle_results"
os.makedirs(output_dir, exist_ok=True)

# Step 2: Sab benchmark results ko ek structured dictionary mein daalo
final_results = {
    "metadata": {
        "model": "google/vit-base-patch16-224",
        "total_parameters": 86567656,
        "gpu": "Tesla T4",
        "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "notebook": "inferbench-vit-gpu-benchmark"
    },
    "pytorch_baseline": pytorch_results,
    "onnx_runtime_cuda": onnx_cuda_results,
    "tensorrt_fp16": trt_results,
    "accuracy_validation": {
        "test_set": "tiny-imagenet validation, 100 samples",
        "onnx_vs_pytorch_match_pct": float(onnx_match),
        "tensorrt_vs_pytorch_match_pct": float(trt_match),
        "onnx_vs_pytorch_mean_abs_diff": float(np.mean(np.abs(pytorch_output - onnx_output))),
        "tensorrt_vs_pytorch_mean_abs_diff": float(np.mean(np.abs(pytorch_output - trt_output))),
        "mismatched_samples": mismatches.tolist()
    },
    "speedup_summary": {
        f"batch_{pytorch_results[i]['batch_size']}": {
            "pytorch_ms": pytorch_results[i]["mean_ms"],
            "onnx_ms": onnx_cuda_results[i]["mean_ms"],
            "tensorrt_ms": trt_results[i]["mean_ms"],
            "tensorrt_speedup_vs_pytorch": round(pytorch_results[i]["mean_ms"] / trt_results[i]["mean_ms"], 2)
        }
        for i in range(len(pytorch_results))
    },
    "environment_issues_resolved": [
        "torchaudio CUDA version mismatch (13.0 vs 12.8) caused transformers import crash - fixed by uninstalling torchaudio",
        "New PyTorch dynamo-based ONNX exporter required 'onnxscript' package - installed separately",
        "Generic 'tensorrt' pip package auto-selected CUDA 13 variant incompatible with onnxruntime-gpu (CUDA 12.x) - fixed by explicitly installing tensorrt-cu12",
        "TensorRT shared libraries (.so files) not in system LD_LIBRARY_PATH - resolved by symlinking all tensorrt_libs/*.so* files into /usr/lib and running ldconfig"
    ]
}

# Step 3: JSON file save karo results folder mein
json_path = os.path.join(output_dir, "benchmark_results.json")
with open(json_path, "w") as f:
    json.dump(final_results, f, indent=2)

print(f"JSON saved: {json_path}")

# Step 4: ONNX model file bhi copy karo (agar future reference ke liye chahiye)
if os.path.exists("vit_base.onnx"):
    shutil.copy("vit_base.onnx", os.path.join(output_dir, "vit_base.onnx"))
    print("ONNX model copied.")

# Step 5: Sab kuch ek zip file mein compress karo
zip_filename = "inferbench_kaggle_results"
shutil.make_archive(zip_filename, 'zip', output_dir)

print(f"\nZip created: {zip_filename}.zip")
print(f"Size: {os.path.getsize(zip_filename + '.zip') / 1024:.1f} KB")

JSON saved: inferbench_kaggle_results/benchmark_results.json
ONNX model copied.

Zip created: inferbench_kaggle_results.zip
Size: 16.4 KB
